<a href="https://colab.research.google.com/github/theboogeyman81/for_deep_learning/blob/main/customer_chrun_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import pickle
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('Churn_Modelling.csv')

In [ ]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
columns_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df_processed = df.drop(columns=columns_to_drop)

In [7]:
df_encoded = pd.get_dummies(df_processed, columns=['Geography', 'Gender'], drop_first=True, dtype=int)

In [8]:
print(f"Features after encoding: {df_encoded.columns.tolist()}")
print(f"Total features: {len(df_encoded.columns) - 1}")  # -1 for target

Features after encoding: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']
Total features: 11


In [9]:
X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

In [10]:
print(f"\n✅ Feature matrix shape: {X.shape}")
print(f"✅ Target vector shape: {y.shape}")


✅ Feature matrix shape: (10000, 11)
✅ Target vector shape: (10000,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train churn rate: {y_train.mean()*100:.2f}%")
print(f"Test churn rate: {y_test.mean()*100:.2f}%")

Training set: 8000 samples
Test set: 2000 samples
Train churn rate: 20.38%
Test churn rate: 20.35%


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
with open('churn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved as 'churn_scaler.pkl'")

✅ Scaler saved as 'churn_scaler.pkl'


In [15]:
feature_names = X.columns.tolist()
with open('churn_feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)
print("✅ Feature names saved")


✅ Feature names saved


In [17]:
print("BUILDING IMPROVED NEURAL NETWORK...")

model = Sequential([
    # Input layer
    Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]), # Removed name='input_layer'
    BatchNormalization(name='batch_norm_1'),
    Dropout(0.3, name='dropout_1'),

    # Hidden layer 1
    Dense(32, activation='relu', name='hidden_layer_1'),
    BatchNormalization(name='batch_norm_2'),
    Dropout(0.3, name='dropout_2'),

    # Hidden layer 2
    Dense(16, activation='relu', name='hidden_layer_2'),
    BatchNormalization(name='batch_norm_3'),
    Dropout(0.2, name='dropout_3'),

    # Output layer (binary classification)
    Dense(1, activation='sigmoid', name='output_layer')
], name='Churn_Prediction_Model')

model.summary()

BUILDING IMPROVED NEURAL NETWORK...


Model: "Churn_Prediction_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_1 (Dense)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_2                    │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_3                    │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,841 (15.00 KB)

 Trainable params: 3,617 (14.13 KB)

 Non-trainable params: 224 (896.00 B)

In [18]:
model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.AUC(name='auc')]
)

In [19]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_churn_model.h5',
        monitor='val_auc',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        verbose=1,
        min_lr=1e-7
    )
]

In [20]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
188/200 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6047 - auc: 0.5987 - loss: 0.7390 - precision: 0.2693 - recall: 0.5054
Epoch 1: val_auc improved from inf to 0.76383, saving model to best_churn_model.h5


200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.6084 - auc: 0.6001 - loss: 0.7336 - precision: 0.2705 - recall: 0.5017 - val_accuracy: 0.8188 - val_auc: 0.7638 - val_loss: 0.4474 - val_precision: 0.7273 - val_recall: 0.1500 - learning_rate: 0.0010
Epoch 2/100
188/200 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7686 - auc: 0.6985 - loss: 0.5072 - precision: 0.4353 - recall: 0.3410
Epoch 2: val_auc did not improve from 0.76383
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7689 - auc: 0.6982 - loss: 0.5068 - precision: 0.4349 - recall: 0.3389 - val_accuracy: 0.8369 - val_auc: 0.7961 - val_loss: 0.4030 - val_precision: 0.7921 - val_recall: 0.2500 - learning_rate: 0.0010
Epoch 3/100
198/200 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7988 - auc: 0.7105 - loss: 0.4660 - precision: 0.5113 - recall: 0.2712
Epoch 3: val_auc did not improve from 0.76383
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7988 - auc: 0.7106 - loss: 0.4661 - precision: 0.5114 - recall: 0.2

In [22]:
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
plt.title('Confusion Matrix - Customer Churn Prediction', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('churn_confusion_matrix.png', dpi=150, bbox_inches='tight')
print("Saved churn_confusion_matrix.png")
plt.close()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Saved churn_confusion_matrix.png
